# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [ ]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [ ]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [ ]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad

NameError: name 'torch' is not defined

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [ ]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [ ]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [ ]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [ ]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [ ]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [ ]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [ ]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)


## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [ ]:
import torch
import torch.nn as nn


class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size

        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_c = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(
        self,
        x: torch.Tensor,
        hc: tuple[torch.Tensor, torch.Tensor] | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if hc is None:
            h = torch.zeros(x.size(0), self.hidden_size, device=x.device, dtype=x.dtype)
            c = torch.zeros(x.size(0), self.hidden_size, device=x.device, dtype=x.dtype)
        else:
            h, c = hc

        combined = torch.cat([x, h], dim=-1)

        f = torch.sigmoid(self.W_f(combined))
        c_tilde = torch.tanh(self.W_c(combined))
        i = torch.sigmoid(self.W_i(combined))
        o = torch.sigmoid(self.W_o(combined))

        c_new = f * c + i * c_tilde
        h_new = o * torch.tanh(c_new)

        return h_new, c_new


cell = LSTMCell(input_size=10, hidden_size=20)
x = torch.randn(3, 10)
h, c = cell(x)
print("h shape:", h.shape, "c shape:", c.shape)

h2, c2 = cell(x, (h, c))
print("h2 shape:", h2.shape, "c2 shape:", c2.shape)

### Inception

![inception](assets/inception.png)

In [ ]:
import torch
import torch.nn as nn


class InceptionModule(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_1x1: int,
        reduce_3x3: int,
        out_3x3: int,
        reduce_5x5: int,
        out_5x5: int,
        out_pool: int,
    ):
        super().__init__()

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.ReLU(),
        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_3x3, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(reduce_3x3, out_3x3, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, reduce_5x5, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(reduce_5x5, out_5x5, kernel_size=5, padding=2),
            nn.ReLU(),
        )

        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1),
            nn.ReLU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        b4 = self.branch4(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


inception = InceptionModule(
    in_channels=192,
    out_1x1=64, reduce_3x3=96, out_3x3=128,
    reduce_5x5=16, out_5x5=32, out_pool=32,
)
x = torch.randn(2, 192, 28, 28)
out = inception(x)
print("Inception output shape:", out.shape)

### SE

![se](assets/SqueezeAndExcite.png)

In [ ]:
import torch
import torch.nn as nn


class SqueezeAndExcitation(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        s = self.squeeze(x).view(b, c)
        e = self.excitation(s).view(b, c, 1, 1)
        return x * e


se = SqueezeAndExcitation(channels=64, reduction=16)
x = torch.randn(2, 64, 32, 32)
out = se(x)
print("SE output shape:", out.shape)

### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [ ]:
import torch
import torch.nn as nn


class SelectiveKernel(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, reduction: int = 16):
        super().__init__()

        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )
        self.conv5 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )

        mid = max(out_channels // reduction, 32)
        self.fc = nn.Sequential(
            nn.Linear(out_channels, mid, bias=False),
            nn.ReLU(),
        )
        self.fc_a = nn.Linear(mid, out_channels, bias=False)
        self.fc_b = nn.Linear(mid, out_channels, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u_tilde = self.conv3(x)
        u_hat = self.conv5(x)

        u = u_tilde + u_hat
        s = u.mean(dim=(-2, -1))  # global average pooling
        z = self.fc(s)

        a = self.fc_a(z)
        b = self.fc_b(z)
        ab = torch.stack([a, b], dim=1)  # (B, 2, C)
        ab = torch.softmax(ab, dim=1)

        a_w = ab[:, 0, :].unsqueeze(-1).unsqueeze(-1)
        b_w = ab[:, 1, :].unsqueeze(-1).unsqueeze(-1)

        return u_tilde * a_w + u_hat * b_w


sk = SelectiveKernel(in_channels=64, out_channels=64)
x = torch.randn(2, 64, 32, 32)
out = sk(x)
print("SelectiveKernel output shape:", out.shape)

### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [ ]:
import torch
import torch.nn as nn


class PatchMerger(nn.Module):
    def __init__(self, dim: int, num_output_tokens: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.queries = nn.Parameter(torch.randn(num_output_tokens, dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, N, D)
        x = self.norm(x)
        # Learned weights (M, D) @ x^T (D, N) → (M, N) per batch
        scores = torch.matmul(self.queries, x.transpose(-1, -2))  # (B, M, N)
        attn = torch.softmax(scores, dim=-1)  # (B, M, N)
        out = torch.matmul(attn, x)  # (B, M, D)
        return out


pm = PatchMerger(dim=128, num_output_tokens=8)
x = torch.randn(2, 64, 128)  # (B, N=64, D=128)
out = pm(x)
print("PatchMerger output shape:", out.shape)  # (2, 8, 128)

## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(x + self.block(x))


res = ResidualBlock(channels=64)
x = torch.randn(2, 64, 16, 16)
out = res(x)
print("ResidualBlock output shape:", out.shape)

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [ ]:
class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, padding: int = 0):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels,
            kernel_size=kernel_size, padding=padding, groups=in_channels, bias=False,
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x


sep = SeparableConv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
x = torch.randn(2, 64, 32, 32)
out = sep(x)
print("SeparableConv2d output shape:", out.shape)

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [ ]:
from typing import Optional
import torch
from torch import nn
import numpy as np


class VanillaAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.W_align = nn.Linear(dim, dim, bias=False)
        self.W_value = nn.Linear(dim, dim, bias=False)
        self.W_query = nn.Linear(dim, dim, bias=False)

    def forward(self, query: torch.Tensor, key: torch.Tensor) -> torch.Tensor:
        # query: (B, d), key: (B, L, d)
        aligned = self.W_align(query)  # (B, d)
        score = torch.bmm(key, aligned.unsqueeze(-1)).squeeze(-1)  # (B, L)
        att = torch.softmax(score, dim=1)  # (B, L)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)  # (B, d)
        out = torch.tanh(self.W_value(context) + self.W_query(query))  # (B, d)
        return out


va = VanillaAttention(dim=32)
query = torch.randn(4, 32)
key = torch.randn(4, 10, 32)
out = va(query, key)
print("VanillaAttention output shape:", out.shape)

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [ ]:
import math


class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # q: (B, L_q, d_k), k: (B, L_k, d_k), v: (B, L_k, d_v)
        d_k = q.size(-1)
        scores = torch.bmm(q, k.transpose(-1, -2)) / math.sqrt(d_k)  # (B, L_q, L_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        return torch.bmm(attn, v)  # (B, L_q, d_v)


sdpa = ScaledDotProductAttention()
q = torch.randn(2, 5, 64)
k = torch.randn(2, 10, 64)
v = torch.randn(2, 10, 64)
out = sdpa(q, k, v)
print("ScaledDotProductAttention output shape:", out.shape)

## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        B = q.size(0)

        q = self.W_q(q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(k).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(v).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        # q, k, v: (B, num_heads, L, d_k)

        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)

        out = torch.matmul(attn, v)  # (B, num_heads, L_q, d_k)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(out)


mha = MultiHeadAttention(d_model=64, num_heads=8)
x = torch.randn(2, 10, 64)
out = mha(x, x, x)
print("MultiHeadAttention output shape:", out.shape)

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [ ]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None) -> torch.Tensor:
        # Pre-LN Transformer Encoder
        h = self.norm1(x)
        x = x + self.attn(h, h, h, mask)
        x = x + self.mlp(self.norm2(x))
        return x


enc = TransformerEncoderLayer(d_model=64, num_heads=8, d_ff=256, dropout=0.1)
x = torch.randn(2, 10, 64)
out = enc(x)
print("TransformerEncoderLayer output shape:", out.shape)


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [ ]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches: int, d_model: int, d_token_hidden: int, d_channel_hidden: int):
        super().__init__()

        self.norm1 = nn.LayerNorm(d_model)
        self.token_mixing = nn.Sequential(
            nn.Linear(num_patches, d_token_hidden),
            nn.GELU(),
            nn.Linear(d_token_hidden, num_patches),
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.channel_mixing = nn.Sequential(
            nn.Linear(d_model, d_channel_hidden),
            nn.GELU(),
            nn.Linear(d_channel_hidden, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, num_patches, d_model)
        h = self.norm1(x).transpose(1, 2)   # (B, d_model, num_patches)
        x = x + self.token_mixing(h).transpose(1, 2)
        x = x + self.channel_mixing(self.norm2(x))
        return x


mixer = MLPMixerBlock(num_patches=16, d_model=64, d_token_hidden=128, d_channel_hidden=256)
x = torch.randn(2, 16, 64)
out = mixer(x)
print("MLPMixerBlock output shape:", out.shape)


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [ ]:
class ConvMixerBlock(nn.Module):
    def __init__(self, dim: int, kernel_size: int = 9):
        super().__init__()
        self.depthwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim),
            nn.ReLU(),
            nn.BatchNorm2d(dim),
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.depthwise(x)  # residual connection
        x = self.pointwise(x)
        return x


cm = ConvMixerBlock(dim=128, kernel_size=9)
x = torch.randn(2, 128, 16, 16)
out = cm(x)
print("ConvMixerBlock output shape:", out.shape)


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

По сути все три подхода занимаются одним и тем же: смешивают информацию между токенами и между каналами. Просто делают это по-разному.

В MHA для смешивания токенов строится матрица внимания $A = \text{softmax}(QK^T / \sqrt{d_k})$, которая зависит от конкретного входа, т.е. для каждого батча она своя. MLPMixer вместо этого просто транспонирует вход и прогоняет через MLP, получается тоже смешивание токенов, но с фиксированными весами. ConvMixer делает то же самое через depthwise свёртку с большим ядром, тоже фиксированные веса, только ещё и локальные.

Если записать формально, для $X \in \mathbb{R}^{L \times D}$ общая схема такая:

$$X' = X + f_{\text{token}}(X), \quad X'' = X' + f_{\text{channel}}(X')$$

Для MHA: $f_{\text{token}}(X) = \text{softmax}\!\left(\frac{XW_Q(XW_K)^T}{\sqrt{d_k}}\right) XW_V W_O$

Для MLPMixer: $f_{\text{token}}(X) = (\sigma(X^T W_1)W_2)^T$

Для ConvMixer: $f_{\text{token}}(X) = \text{DepthwiseConv}(X)$

Каналы во всех трёх случаях смешиваются линейным слоем (FFN / MLP / pointwise conv).

То есть матрица внимания в MHA это по сути матрица смешивания, которая каждый раз пересчитывается под вход. А в MLPMixer и ConvMixer эта матрица просто фиксированная и обучается один раз. Оказывается, что если данных достаточно, то фиксированных паттернов хватает и адаптивность attention не даёт большого выигрыша.

Про плюсы/минусы: MHA видит все токены сразу (глобальное поле), веса адаптивные, но сложность квадратичная $O(L^2 D)$ и без предобучения на большом датасете работает так себе, нет inductive bias. MLPMixer тоже глобальный и попроще в вычислениях, но у него длина входа $L$ жёстко вшита в размер MLP, нельзя просто так подать последовательность другой длины. ConvMixer самый дешёвый по вычислениям (линейно от $L$), хорошо работает на маленьких датасетах за счёт локальности и трансляционной инвариантности, но рецептивное поле ограничено размером ядра.